# 🌍 CO₂ Emissions Analysis — Environmental Economics Assignment
**Dataset:** Our World in Data CO₂ Emissions  
**Plots:** 9 publication-quality visualisations covering economic, temporal, geographic and policy dimensions.

---
### ⚙️ Setup instructions
Before running, install required libraries (run once):
```bash
pip install pandas numpy matplotlib seaborn plotly kaleido
```
Place `owid-co2-data.csv` in the **same folder as this notebook**, then run all cells.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Plotly for the interactive choropleth (Plot 7)
try:
    import plotly.express as px
    import plotly.io as pio
    pio.renderers.default = "notebook"
    HAS_PLOTLY = True
    print("✓ Plotly available — Plot 7 will render as interactive choropleth map")
except ImportError:
    HAS_PLOTLY = False
    print("⚠ Plotly not found — Plot 7 will fall back to ranked bar chart")
    print("  Install with: pip install plotly")

%matplotlib inline
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    14,
    "axes.titleweight":  "bold",
    "axes.labelsize":    12,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        120,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
    "savefig.facecolor": "white",
    "legend.framealpha": 0.85,
})

# ── Constants ──────────────────────────────────────────────────────────────────
HIGHLIGHT = {
    "United States":       "#E63946",
    "China":               "#F4A261",
    "India":               "#2A9D8F",
    "Brazil":              "#457B9D",
    "European Union (27)": "#8338EC",
}

# Output folder for saved PNGs (created next to the notebook)
OUT = Path("co2_plots")
OUT.mkdir(exist_ok=True)
print(f"\n✓ Plots will be saved to: {OUT.resolve()}")

## 📥 Load & Clean Data

In [ ]:
CSV_PATH = "owid-co2-data.csv"   # ← change path here if the file is elsewhere

df = pd.read_csv(CSV_PATH, low_memory=False)

# Derive GDP per capita (not pre-computed in the file)
if "gdp_per_capita" not in df.columns:
    df["gdp_per_capita"] = df["gdp"] / df["population"]

# country_df: only rows with a real ISO-3 code (drops regional aggregates)
country_df = df[df["iso_code"].notna() & (df["iso_code"] != "")].copy()

print(f"Full dataset : {df.shape[0]:,} rows  |  years {df['year'].min()}–{df['year'].max()}")
print(f"Country rows : {country_df.shape[0]:,} rows  |  {country_df['country'].nunique()} unique countries")
df.head(3)

---
## Plot 1 — GDP per Capita vs CO₂ per Capita
> **What it shows:** The relationship between national wealth and per-person carbon footprint.  
> **Economics angle:** Tests the Environmental Kuznets Curve hypothesis. Richer countries generally emit more, but European nations achieve lower emissions at similar GDP levels — pointing to policy and energy-mix differences.

In [ ]:
df_valid = country_df.dropna(subset=["gdp_per_capita", "co2_per_capita"])
df_valid = df_valid[(df_valid["gdp_per_capita"] > 0) & (df_valid["co2_per_capita"] > 0)]
latest   = df_valid["year"].max()
df1      = df_valid[df_valid["year"] == latest][["country","gdp_per_capita","co2_per_capita","population"]].copy()

fig, ax = plt.subplots(figsize=(11, 7))
sizes   = np.clip(df1["population"] / 1e6 * 0.4, 10, 600)

ax.scatter(df1["gdp_per_capita"], df1["co2_per_capita"],
           s=sizes, alpha=0.50, color="#4A90D9", edgecolors="white", linewidths=0.4)

for country, colour in HIGHLIGHT.items():
    row = df1[df1["country"] == country]
    if row.empty: continue
    ax.scatter(row["gdp_per_capita"], row["co2_per_capita"],
               s=200, color=colour, edgecolors="white", linewidths=1.2, zorder=5)
    ax.annotate(country, (row["gdp_per_capita"].values[0], row["co2_per_capita"].values[0]),
                xytext=(6, 3), textcoords="offset points",
                fontsize=8.5, color=colour, fontweight="bold")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("GDP per Capita (USD, log scale)")
ax.set_ylabel("CO₂ per Capita (tonnes, log scale)")
ax.set_title(f"GDP per Capita vs CO₂ per Capita ({latest})\nBubble size ∝ population")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(handles=[mpatches.Patch(color=c, label=n) for n, c in HIGHLIGHT.items()],
          title="Highlighted Countries", loc="upper left", fontsize=8.5)

plt.tight_layout()
plt.savefig(OUT / "plot1_gdp_vs_co2.png")
plt.show()
print(f"✓ Saved → {OUT}/plot1_gdp_vs_co2.png")

---
## Plot 2 — CO₂ per GDP over Time (Carbon Efficiency of Economies)
> **What it shows:** How many kg of CO₂ are emitted per dollar of economic output over time.  
> **Economics angle:** All major economies improved efficiency, but absolute emissions still rose — a classic "rebound effect". China's sharp drop after 2000 reflects industrial modernisation.

In [ ]:
countries2 = ["United States","China","India","Germany","United Kingdom","Japan","Brazil","Russia"]
palette2   = sns.color_palette("tab10", len(countries2))

fig, ax = plt.subplots(figsize=(12, 6))
for country, colour in zip(countries2, palette2):
    sub = (country_df[country_df["country"] == country][["year","co2_per_gdp"]]
           .dropna().query("year >= 1970"))
    if sub.empty: continue
    ax.plot(sub["year"], sub["co2_per_gdp"], label=country, color=colour, linewidth=2.0)

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ per GDP (kg CO₂ per constant $)")
ax.set_title("Carbon Intensity of GDP over Time\n(Lower = more carbon-efficient economy)")
ax.legend(fontsize=9, ncol=2)
ax.set_xlim(1970, country_df["year"].max())

plt.tight_layout()
plt.savefig(OUT / "plot2_carbon_efficiency.png")
plt.show()
print(f"✓ Saved → {OUT}/plot2_carbon_efficiency.png")

---
## Plot 3 — Production vs Consumption vs Trade CO₂
> **What it shows:** Whether a country's production-based emissions match its consumption footprint.  
> **Economics angle:** Positive trade CO₂ means a country imports embodied carbon — rich nations effectively outsource emissions to manufacturing hubs like China. This is central to "carbon leakage" debates in climate policy.

In [ ]:
economies3 = ["United States","China","India","Germany","United Kingdom","Japan","Russia","Brazil"]

rows3 = []
for e in economies3:
    sub = country_df[country_df["country"] == e].sort_values("year", ascending=False)
    for _, row in sub.iterrows():
        if pd.notna(row["co2"]) and pd.notna(row["consumption_co2"]) and pd.notna(row["trade_co2"]):
            rows3.append({"country": e, "production": row["co2"],
                          "consumption": row["consumption_co2"], "trade": row["trade_co2"]})
            break

df3   = pd.DataFrame(rows3)
x3    = np.arange(len(df3))
w     = 0.26

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x3 - w, df3["production"],  w, label="Production CO₂",  color="#E63946", alpha=0.85)
ax.bar(x3,     df3["consumption"], w, label="Consumption CO₂", color="#457B9D", alpha=0.85)
ax.bar(x3 + w, df3["trade"],       w, label="Trade CO₂",       color="#2A9D8F", alpha=0.85)

ax.set_xticks(x3)
ax.set_xticklabels(df3["country"], rotation=20, ha="right")
ax.set_ylabel("CO₂ Emissions (Mt)")
ax.set_title("Production vs Consumption vs Trade CO₂\nfor Major Economies (most recent complete year)")
ax.legend()
ax.axhline(0, color="black", linewidth=0.7, linestyle="--")

plt.tight_layout()
plt.savefig(OUT / "plot3_prod_vs_consumption.png")
plt.show()
print(f"✓ Saved → {OUT}/plot3_prod_vs_consumption.png")

---
## Plot 4 — Fuel-Source Breakdown for Top 10 Emitters
> **What it shows:** The mix of coal, oil, gas, cement and flaring driving each country's emissions.  
> **Economics angle:** Coal dominance in China and India means a coal phase-out is the single highest-leverage intervention globally. The US's gas/oil balance reflects a different structural challenge.

In [ ]:
fuels4   = ["coal_co2","oil_co2","gas_co2","cement_co2","flaring_co2"]
colours4 = ["#264653","#E9C46A","#F4A261","#E76F51","#A8DADC"]

# Use most recent year that has all fuel columns
df4_all = country_df.dropna(subset=fuels4)
latest4 = df4_all["year"].max()
df4     = df4_all[df4_all["year"] == latest4][["country"] + fuels4].copy()
df4["total"] = df4[fuels4].sum(axis=1)
top10   = df4.nlargest(10, "total").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 6))
bottom  = np.zeros(len(top10))
for fuel, colour in zip(fuels4, colours4):
    vals = top10[fuel].fillna(0).values
    ax.bar(top10["country"], vals, bottom=bottom,
           color=colour, label=fuel.replace("_co2","").capitalize(), alpha=0.9)
    bottom += vals

ax.set_xticklabels(top10["country"], rotation=25, ha="right")
ax.set_ylabel("CO₂ Emissions (Mt)")
ax.set_title(f"Fuel-Source Breakdown — Top 10 Emitters ({latest4})")
ax.legend(title="Fuel source", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig(OUT / "plot4_fuel_breakdown.png")
plt.show()
print(f"✓ Saved → {OUT}/plot4_fuel_breakdown.png")

---
## Plot 5 — Carbon Intensity of the Energy Mix over Time
> **What it shows:** CO₂ emitted per unit of energy consumed — a measure of how clean the grid is.  
> **Economics angle:** France (nuclear) and Brazil (hydro) maintain low intensity. Countries still high on this metric need structural energy transition, not just efficiency gains.

In [ ]:
countries5 = ["United States","China","India","France","Germany","United Kingdom","Japan","Brazil"]
palette5   = sns.color_palette("Set2", len(countries5))

fig, ax = plt.subplots(figsize=(12, 6))
for country, colour in zip(countries5, palette5):
    sub = (country_df[(country_df["country"] == country) & (country_df["year"] >= 1970)]
           [["year","co2_per_unit_energy"]].dropna())
    if sub.empty: continue
    ax.plot(sub["year"], sub["co2_per_unit_energy"], label=country, color=colour, linewidth=2.0)

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ per Unit Energy (kg CO₂ per kWh)")
ax.set_title("Carbon Intensity of the Energy Mix over Time\n(Lower = cleaner energy)")
ax.legend(fontsize=9, ncol=2)

plt.tight_layout()
plt.savefig(OUT / "plot5_energy_intensity.png")
plt.show()
print(f"✓ Saved → {OUT}/plot5_energy_intensity.png")

---
## Plot 6 — Land-Use Change CO₂ (Brazil, Indonesia, DR Congo)
> **What it shows:** Emissions from deforestation and land conversion over time.  
> **Economics angle:** Brazil's peak coincides with Amazon agricultural expansion; the subsequent drop after 2004 demonstrates that REDD+ style policies can work. These emissions are often excluded from industrial CO₂ accounting but are critical for net-zero targets.

In [ ]:
countries6 = ["Brazil","Indonesia","Democratic Republic of Congo"]
colours6   = ["#2D6A4F","#D62828","#F77F00"]

fig, ax = plt.subplots(figsize=(12, 6))
for country, colour in zip(countries6, colours6):
    sub = (country_df[(country_df["country"] == country) & (country_df["year"] >= 1850)]
           [["year","land_use_change_co2"]].dropna())
    if sub.empty: continue
    ax.plot(sub["year"], sub["land_use_change_co2"], label=country, color=colour, linewidth=2.2)
    ax.fill_between(sub["year"], sub["land_use_change_co2"], alpha=0.12, color=colour)

ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_xlabel("Year")
ax.set_ylabel("Land-Use Change CO₂ (Mt)")
ax.set_title("Land-Use Change CO₂ Emissions\nBrazil · Indonesia · DR Congo")
ax.legend()

plt.tight_layout()
plt.savefig(OUT / "plot6_land_use.png")
plt.show()
print(f"✓ Saved → {OUT}/plot6_land_use.png")

---
## Plot 7 — World Choropleth Map of CO₂ per Capita
> **What it shows:** Geographic distribution of per-person emissions.  
> **Economics angle:** Gulf states and Australia dominate per-capita rankings despite small populations — pointing to fossil-fuel-dependent economies. Large populous countries like India and much of Africa have very low per-capita footprints, illustrating the equity dimension of climate policy.

*Interactive map if Plotly is installed; otherwise a ranked bar chart.*

In [ ]:
df7_all = country_df.dropna(subset=["co2_per_capita"])
df7_all = df7_all[df7_all["co2_per_capita"] > 0]
latest7 = df7_all["year"].max()
df7     = df7_all[df7_all["year"] == latest7][["country","iso_code","co2_per_capita"]].copy()

if HAS_PLOTLY:
    fig7 = px.choropleth(
        df7,
        locations="iso_code",
        color="co2_per_capita",
        hover_name="country",
        color_continuous_scale="YlOrRd",
        range_color=(0, df7["co2_per_capita"].quantile(0.95)),
        labels={"co2_per_capita": "CO₂ per Capita (t)"},
        title=f"CO₂ per Capita by Country ({latest7})",
    )
    fig7.update_layout(margin=dict(l=0, r=0, t=40, b=0), height=500)
    fig7.write_image(str(OUT / "plot7_choropleth.png"), width=1400, height=700, scale=2)
    fig7.show()
    print(f"✓ Saved → {OUT}/plot7_choropleth.png")
else:
    # Fallback: ranked horizontal bar chart
    top30 = df7.nlargest(30, "co2_per_capita").reset_index(drop=True)
    cmap7 = plt.cm.get_cmap("YlOrRd", 30)
    cols7 = [cmap7(i / 30) for i in range(len(top30))]

    fig, ax = plt.subplots(figsize=(13, 8))
    ax.barh(top30["country"][::-1], top30["co2_per_capita"][::-1],
            color=cols7[::-1], edgecolor="white", linewidth=0.4)

    sm = plt.cm.ScalarMappable(cmap="YlOrRd",
                               norm=plt.Normalize(0, top30["co2_per_capita"].max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="vertical", fraction=0.02, pad=0.02)
    cbar.set_label("CO₂ per Capita (tonnes)")

    ax.set_xlabel("CO₂ per Capita (tonnes)")
    ax.set_title(f"Top 30 Countries by CO₂ per Capita ({latest7})\n"
                 "(Install plotly for interactive choropleth map)")
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)

    plt.tight_layout()
    plt.savefig(OUT / "plot7_world_map.png")
    plt.show()
    print(f"✓ Saved → {OUT}/plot7_world_map.png")

---
## Plot 8 — Historical Responsibility vs Current Per-Capita Emissions
> **What it shows:** Countries plotted by cumulative (all-time) CO₂ on x-axis and today's per-capita emissions on y-axis.  
> **Economics angle:** The "climate justice" plot. Countries in the top-right (high on both axes, e.g. USA, Australia) bear the greatest moral and economic obligation for climate finance. India — low historical, low per-capita — is a classic CBDR-RC example.

In [ ]:
df8_valid = country_df.dropna(subset=["cumulative_co2","co2_per_capita"])
df8_valid = df8_valid[(df8_valid["cumulative_co2"] > 0) & (df8_valid["co2_per_capita"] > 0)]
latest8   = df8_valid["year"].max()
df8       = df8_valid[df8_valid["year"] == latest8][["country","cumulative_co2","co2_per_capita","population"]].copy()

label_list8 = ["United States","China","Russia","Germany","United Kingdom",
               "India","Japan","Canada","Australia","Brazil",
               "France","South Africa","South Korea","Saudi Arabia"]

fig, ax = plt.subplots(figsize=(12, 7))
sizes8  = np.clip(df8["population"] / 1e6 * 0.5, 8, 800)
ax.scatter(df8["cumulative_co2"], df8["co2_per_capita"],
           s=sizes8, alpha=0.45, color="#6B9BC3", edgecolors="white", linewidths=0.3)

for country in label_list8:
    row = df8[df8["country"] == country]
    if row.empty: continue
    col = HIGHLIGHT.get(country, "#333333")
    ax.scatter(row["cumulative_co2"], row["co2_per_capita"],
               s=170, color=col, edgecolors="white", linewidths=1, zorder=5)
    ax.annotate(country, (row["cumulative_co2"].values[0], row["co2_per_capita"].values[0]),
                xytext=(5, 3), textcoords="offset points",
                fontsize=8, color=col, fontweight="bold")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Cumulative CO₂ Emissions (Mt, log scale) — Historical Responsibility")
ax.set_ylabel("CO₂ per Capita today (tonnes, log scale) — Current Lifestyle")
ax.set_title(f"Historical Responsibility vs Current Per-Capita Emissions ({latest8})\nBubble size ∝ population")

plt.tight_layout()
plt.savefig(OUT / "plot8_historical_vs_current.png")
plt.show()
print(f"✓ Saved → {OUT}/plot8_historical_vs_current.png")

---
## Plot 9 — Global CO₂ Trend with Policy Milestones
> **What it shows:** The full arc of global industrial CO₂ emissions since 1850, annotated with key international climate agreements.  
> **Economics angle:** Emissions continued rising through every major agreement — illustrating that voluntary pledges without enforcement or pricing mechanisms are insufficient. The post-2015 slowdown is modest; the gap to net-zero remains enormous.

In [ ]:
world9 = df[df["country"] == "World"][["year","co2"]].dropna()
if world9.empty:
    world9 = country_df.groupby("year", as_index=False)["co2"].sum()
world9 = world9[world9["year"] >= 1850]

milestones = {
    1972: ("Stockholm\nConference", "top"),
    1992: ("UNFCCC\nRio",           "bottom"),
    1997: ("Kyoto\nProtocol",       "top"),
    2005: ("Kyoto\nIn Force",       "bottom"),
    2015: ("Paris\nAgreement",      "top"),
    2021: ("COP26\nGlasgow",        "bottom"),
}

fig, ax = plt.subplots(figsize=(13, 6))
ax.fill_between(world9["year"], world9["co2"], alpha=0.15, color="#E63946")
ax.plot(world9["year"], world9["co2"], color="#E63946", linewidth=2.2, label="Global CO₂ (Mt)")

ymax = world9["co2"].max()
for yr, (label, pos) in milestones.items():
    ax.axvline(yr, color="#555555", linewidth=1.0, linestyle="--", alpha=0.7)
    y_text = ymax * 0.93 if pos == "top" else ymax * 0.12
    ax.text(yr, y_text, f"{yr}\n{label}",
            ha="center", va="top" if pos == "top" else "bottom",
            fontsize=7.5, color="#333333",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#AAAAAA", alpha=0.85))

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ Emissions (Mt)")
ax.set_title("Global CO₂ Emissions Trend with Key Policy Milestones")
ax.set_xlim(1850, world9["year"].max() + 2)
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig(OUT / "plot9_global_trend.png")
plt.show()
print(f"✓ Saved → {OUT}/plot9_global_trend.png")

---
## ✅ Summary

All 9 plots have been saved to the `co2_plots/` folder next to this notebook.

| # | Plot | Key Takeaway |
|---|------|--------------|
| 1 | GDP vs CO₂ per Capita | Wealthier countries emit more, but European nations beat the trend |
| 2 | CO₂ per GDP over time | Efficiency is rising but absolute emissions keep growing |
| 3 | Production vs Consumption | Rich nations outsource emissions through trade |
| 4 | Fuel breakdown — top 10 | Coal dominates China/India; structural reform is the key lever |
| 5 | Energy carbon intensity | Nuclear/hydro grids are far cleaner than coal-heavy ones |
| 6 | Land-use change CO₂ | Deforestation is a major and often-ignored emissions source |
| 7 | World map CO₂ per capita | Gulf states & Australia lead per-person emissions |
| 8 | Historical vs current | USA & Australia: high on both responsibility and lifestyle axes |
| 9 | Global trend + milestones | Climate agreements haven't yet bent the emissions curve enough |